# Submission Evaluation

Credits/Reference:

* https://www.kaggle.com/competitions/march-machine-learning-mania-2025/discussion/563189
* https://www.kaggle.com/code/rsa013/march-madness-submission-tester
* https://www.kaggle.com/code/rsa013/march-madness-submission-benchmark-example

## setup

In [1]:
import os

# Move up one level to set the working directory to the repo root
os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [2]:
from typing import Sequence, Literal
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import functools
from itertools import combinations
import re

DEFAULT_COMPETITION_DATA_PATH = Path(
    os.path.abspath(os.path.join(os.getcwd(), r"data\kaggle"))
)
SUBMISSION_DATA_PATH = Path(
    os.path.abspath(os.path.join(os.getcwd(), r"data\submissions"))
)
DATA_PATH = Path(os.path.abspath(os.path.join(os.getcwd(), r"data")))

In [3]:
# Define function to determine gender based on TeamID1
def determine_gender(team_id):
    if str(team_id).startswith("1"):  # Men's teams start with 1
        return "Men"
    elif str(team_id).startswith("3"):  # Women's teams start with 3
        return "Women"
    else:
        return None  # Handle unexpected cases


def extract_game_info(id_str: str) -> tuple[int, int, int]:
    """
    Extract season and team IDs from a Kaggle competition game ID string.

    Args:
        id_str (str): The game ID string formatted as "YYYY_Team1_Team2".

    Returns:
        tuple[int, int, int]: A tuple containing (Season, TeamID1, TeamID2).
    """
    try:
        year, team1, team2 = map(int, id_str.split("_"))
        return year, team1, team2
    except ValueError:
        raise ValueError(f"Unexpected ID format: {id_str}")


### Example usage:
# # Load the sample submission file
# submission_df = pd.read_csv('/mnt/data/SampleSubmissionStage1.csv')

# # Extract game info into new columns
# submission_df[['Season', 'TeamID1', 'TeamID2']] = submission_df['ID'].apply(extract_game_info).apply(pd.Series)


def extract_seed_value(seed_str: str) -> int:
    """
    Extracts the numeric seed value from an NCAA tournament seed string.

    Args:
        seed_str (str): The seed string (e.g., 'W01', 'Y12b').

    Returns:
        int: The extracted seed value, or 16 if extraction fails.
    """
    try:
        # Extract numeric part using regex
        match = re.search(r"\d+", seed_str)
        if match:
            return int(match.group())
        else:
            return 16  # Default seed value for unselected teams/errors
    except (ValueError, TypeError):
        return 16  # Default for unexpected cases


### Example usage:
# # Load the tournament seed data
# w_seed = pd.read_csv('/mnt/data/WNCAATourneySeeds.csv')
# m_seed = pd.read_csv('/mnt/data/MNCAATourneySeeds.csv')

# # Concatenate men's and women's data
# seed_df = pd.concat([m_seed, w_seed], axis=0, ignore_index=True)

# # Apply the function to extract seed values
# seed_df['SeedValue'] = seed_df['Seed'].apply(extract_seed_value)


# Function to merge results_df with seeds_df and tourney_round_lookup
def process_results(results_df, seeds_df, tourney_round_lookup):
    results_df = results_df.merge(
        seeds_df,
        left_on=["Season", "WTeamID"],
        right_on=["Season", "TeamID"],
        how="left",
        suffixes=("", "_T1"),
    )
    results_df = results_df.merge(
        seeds_df,
        left_on=["Season", "LTeamID"],
        right_on=["Season", "TeamID"],
        how="left",
        suffixes=("", "_T2"),
    )

    results_df["WSeed"] = results_df["Seed"]
    results_df["LSeed"] = results_df["Seed_T2"]

    # Ensure StrongSeed is alphabetically first, and WeakSeed is second
    results_df["StrongSeed"] = results_df[["WSeed", "LSeed"]].min(axis=1)
    results_df["WeakSeed"] = results_df[["WSeed", "LSeed"]].max(axis=1)

    results_df = results_df.merge(
        tourney_round_lookup,
        left_on=["StrongSeed", "WeakSeed"],
        right_on=["StrongSeed", "WeakSeed"],
        how="left",
    )

    # Drop unnecessary columns and return
    results_df = results_df[
        [
            "Season",
            "DayNum",
            "WTeamID",
            "WSeed",
            "WScore",
            "LTeamID",
            "LSeed",
            "LScore",
            "WLoc",
            "NumOT",
            "Round",
            "Slot",
        ]
    ]

    return results_df


### Example usage:
# # Load data
# m_seed = pd.read_csv(r"data\kaggle\MNCAATourneySeeds.csv")
# w_seed = pd.read_csv(r"data\kaggle\WNCAATourneySeeds.csv")
# m_results = pd.read_csv(r"data\kaggle\MNCAATourneyCompactResults.csv")
# w_results = pd.read_csv(r"data\kaggle\WNCAATourneyCompactResults.csv")
# tourney_round_lookup = pd.read_csv(r"data\tourney_round_lookup.csv")

# # Extract numeric seed values
# m_seed['SeedValue'] = m_seed['Seed'].apply(extract_seed_value)
# w_seed['SeedValue'] = w_seed['Seed'].apply(extract_seed_value)

# # Merge results_df with seeds_df and tourney_round_lookup
# m_results = process_results(m_results, m_seed, tourney_round_lookup)
# w_results = process_results(w_results, w_seed, tourney_round_lookup)


# Function to merge sub_df with seeds_df and tourney_round_lookup
def process_submission(sub_df, seeds_df, tourney_round_lookup):
    sub_df = sub_df.merge(
        seeds_df,
        left_on=["Season", "TeamID1"],
        right_on=["Season", "TeamID"],
        how="left",
        suffixes=("", "_T1"),
    )
    sub_df = sub_df.merge(
        seeds_df,
        left_on=["Season", "TeamID2"],
        right_on=["Season", "TeamID"],
        how="left",
        suffixes=("", "_T2"),
    )

    sub_df["Seed1"] = sub_df["Seed"].fillna("")
    sub_df["Seed2"] = sub_df["Seed_T2"].fillna("")

    # Ensure StrongSeed is alphabetically first, and WeakSeed is second
    sub_df["StrongSeed"] = sub_df[["Seed1", "Seed2"]].min(axis=1)
    sub_df["WeakSeed"] = sub_df[["Seed1", "Seed2"]].max(axis=1)

    sub_df = sub_df.merge(
        tourney_round_lookup,
        left_on=["StrongSeed", "WeakSeed"],
        right_on=["StrongSeed", "WeakSeed"],
        how="left",
    )

    # Drop unnecessary columns and return
    sub_df = sub_df[
        [
            "ID",
            "Pred",
            "Season",
            "TeamID1",
            "Seed1",
            "TeamID2",
            "Seed2",
            "Round",
            "Slot",
        ]
    ]

    return sub_df


### Example usage:
# # Load data
# m_seed = pd.read_csv(r"data\kaggle\MNCAATourneySeeds.csv")
# w_seed = pd.read_csv(r"data\kaggle\WNCAATourneySeeds.csv")
# submission_df = pd.read_csv(r"data\kaggle\SampleSubmissionStage1.csv")
# tourney_round_lookup = pd.read_csv(r"data\tourney_round_lookup.csv")

# # Extract numeric seed values
# m_seed['SeedValue'] = m_seed['Seed'].apply(extract_seed_value)
# w_seed['SeedValue'] = w_seed['Seed'].apply(extract_seed_value)

# # Extract game info
# submission_df[['Season', 'TeamID1', 'TeamID2']] = submission_df['ID'].apply(extract_game_info).apply(pd.Series)

# # Assign Gender column
# submission_df['Gender'] = submission_df['TeamID1'].apply(determine_gender)

# # Split into men's and women's submission dataframes
# m_submission = submission_df[submission_df['Gender'] == 'Men'].copy()
# w_submission = submission_df[submission_df['Gender'] == 'Women'].copy()

# # Merge sub_df with seeds_df and tourney_round_lookup
# m_submission = process_submission(m_submission, m_seed, tourney_round_lookup)
# w_submission = process_submission(w_submission, w_seed, tourney_round_lookup)

In [4]:
def _get_historical_results(
    seasons: Sequence[int] = (2021, 2022, 2023, 2024),
    genders: Sequence[Literal["Men", "Women"]] = ("Men", "Women"),
    rounds: Sequence[int] = (1, 2, 3, 4, 5, 6),  # Default to all rounds
) -> dict[str, int]:
    """
    Parse a list of historical results to the expected Kaggle result format in a dictionary.
    Allows filtering by season, gender, and round.

    Parameters
    ----------
    seasons : Sequence[int], optional
        The seasons to retrieve results for. Defaults to (2021, 2022, 2023, 2024).
    genders : Sequence[Literal["Men", "Women"]], optional
        Filter for Men's or Women's tournament. Defaults to both.
    rounds : Sequence[int], optional
        If provided, only includes games from the specified rounds. Defaults to all rounds.

    Returns
    -------
    dict[str, int]
        A dictionary mapping game IDs to results (1 for original order win, 0 for swapped order).
    """
    # Determine the gender prefix
    if set(genders) == {"Men", "Women"}:
        gender_prefix = "MW"
    elif "Men" in genders:
        gender_prefix = "M"
    else:
        gender_prefix = "W"

    tourney_df = pd.concat(
        [
            _cached_csv_read(
                DEFAULT_COMPETITION_DATA_PATH / f"{mw}NCAATourneyCompactResults.csv"
            )
            for mw in gender_prefix
        ]
    )

    seeds_df = pd.concat(
        [
            _cached_csv_read(
                DEFAULT_COMPETITION_DATA_PATH / f"{mw}NCAATourneySeeds.csv"
            )
            for mw in gender_prefix
        ]
    )

    # Extract numeric seed values
    seeds_df["SeedValue"] = seeds_df["Seed"].apply(extract_seed_value)

    # Load tournament round lookup
    tourney_round_lookup = _cached_csv_read(DATA_PATH / "tourney_round_lookup.csv")

    # Filter tournament results by season
    tourney_df = tourney_df.loc[
        tourney_df["Season"].isin(seasons), ["Season", "WTeamID", "LTeamID"]
    ].copy()

    # Add Gender
    tourney_df["Gender"] = tourney_df["WTeamID"].apply(determine_gender)

    # Add seeds to identify play-in games
    for wl in "WL":
        tourney_df = pd.merge(
            left=tourney_df,
            right=seeds_df.add_suffix(f"_{wl}")[
                [f"Season_{wl}", f"TeamID_{wl}", f"Seed_{wl}", f"SeedValue_{wl}"]
            ],
            how="left",
            left_on=["Season", f"{wl}TeamID"],
            right_on=[f"Season_{wl}", f"TeamID_{wl}"],
        ).copy()

    # Ensure StrongSeed is alphabetically first, and WeakSeed is second
    tourney_df["StrongSeed"] = (
        tourney_df[["Seed_W", "Seed_L"]].min(axis=1).str.rstrip("ab")
    )
    tourney_df["WeakSeed"] = (
        tourney_df[["Seed_W", "Seed_L"]].max(axis=1).str.rstrip("ab")
    )

    # Add Round and Slot from tourney_round_lookup
    tourney_df = tourney_df.merge(
        tourney_round_lookup, on=["StrongSeed", "WeakSeed"], how="left"
    )

    # Filter out play-in games
    play_in_game_mask = tourney_df["Seed_W"].str.endswith(("a", "b")) & tourney_df[
        "Seed_L"
    ].str.endswith(("a", "b"))
    tourney_df = tourney_df[~play_in_game_mask].copy()

    # Filter by specific rounds if provided
    if rounds is not None:
        tourney_df = tourney_df[tourney_df["Round"].isin(rounds)]

    # tourney_df.to_csv(DATA_PATH / "tourney_df.csv", index=False)

    # Create game IDs
    w_ids = (
        tourney_df["Season"].astype(str)
        + "_"
        + tourney_df["WTeamID"].astype(str)
        + "_"
        + tourney_df["LTeamID"].astype(str)
    )

    # Flip game IDs to ensure lowest TeamID appears first
    dont_flip = (tourney_df["WTeamID"] < tourney_df["LTeamID"]).tolist()
    results = {
        wid if df else _swap_game_id(wid): df for wid, df in zip(w_ids, dont_flip)
    }

    return results

In [5]:
def display_submission_results(sub):
    import pandas as pd

    results = {
        "Total Score": evaluate_stage1_submission(sub),
        "By Year": {
            year: evaluate_stage1_submission(sub, [year])
            for year in [2021, 2022, 2023, 2024]
        },
        "By Gender": {
            gender: evaluate_stage1_submission(sub, genders=[gender])
            for gender in ["Men", "Women"]
        },
        "By Year, Gender": {
            (year, gender): evaluate_stage1_submission(sub, [year], [gender])
            for year in [2021, 2022, 2023, 2024]
            for gender in ["Men", "Women"]
        },
        "By Year, Gender, Round": {
            (year, gender, rnd): evaluate_stage1_submission(
                sub, [year], [gender], [rnd]
            )
            for year in [2021, 2022, 2023, 2024]
            for gender in ["Men", "Women"]
            for rnd in [1, 2, 3, 4, 5, 6]
        },
    }

    for key, value in results.items():
        print(f"{key}: {value}")

    return pd.DataFrame(
        results["By Year, Gender, Round"].items(),
        columns=["(Year, Gender, Round)", "Score"],
    )

## march-madness-submission-tester

Modifications:

* Commented out the section for setting competition path automatically.  I manually set `DEFAULT_COMPETITION_DATA_PATH`.
* In the _get_historical_results() function, play_in_game_mask doesn't seem to be filtering out the play-in games. As an example, '2024_1161_1438' was a play-in game but appears as the first result in `evaluate_stage1_submission_games(sub, [2024])`

I replaced this:
`play_in_game_mask = tourney_df["Seed_W"].map(lambda x: x in ("a", "b")) & tourney_df["Seed_L"].map(lambda x: x in ("a", "b"))`

with this:
`play_in_game_mask = tourney_df["Seed_W"].str.endswith(("a", "b")) & tourney_df["Seed_L"].str.endswith(("a", "b"))`

I also replaced this:
`dont_flip = (tourney_df["WTeamID"] < tourney_df["LTeamID"]).to_list()`

with this:
`dont_flip = (kaggle_results["WTeamID"] < kaggle_results["LTeamID"]).to_list()`

since kaggle_results has the play-in games filtered out and tourney_df does not.

In [6]:
# %% [code]
"""
This utility script provides some useful utilities for benchmarking submissions.

You can find notebook examples of usage here:
https://www.kaggle.com/code/rsa013/march-madness-submission-benchmark-example/edit/run/222853167
"""

from typing import Sequence, Literal
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import functools
from itertools import combinations
import os


# ######### sets competition path automatically #########
# try:
#     from dotenv import load_dotenv

#     load_dotenv()
# except ModuleNotFoundError:
#     pass
# if (p := os.getenv("KAGGLE_KERNEL_RUN_TYPE")) is not None:
#     DEFAULT_COMPETITION_DATA_PATH = Path(
#         "/kaggle/input/march-machine-learning-mania-2025"
#     )
#     # COMPETITION_DATA_PATH = Path("/kaggle/input/march-machine-learning-mania-2025")
# elif (p := os.getenv("COMPETITION_DATA_PATH")) is not None:
#     DEFAULT_COMPETITION_DATA_PATH = Path(p)
# else:
#     raise RuntimeError(
#         "If running locally you must define an environment variable COMPETITION_DATA_PATH."
#     )
# #######################################################


def evaluate_stage1_submission(
    submission: pd.DataFrame,
    seasons: Sequence[int] = (2021, 2022, 2023, 2024),
    genders: Sequence[Literal["Men", "Women"]] = ("Men", "Women"),
    rounds: Sequence[int] = (1, 2, 3, 4, 5, 6),
    mode: Literal["brier", "logloss"] = "brier",
) -> float:
    """
    Evaluate a Stage 1 submission for a forecasting competition.

    Parameters
    ----------
    submission : pd.DataFrame
        The submission DataFrame, containing forecasted probabilities.
    seasons : Sequence[int], optional
        The seasons to evaluate the submission for. Defaults to (2021, 2022, 2023, 2024).
    genders : Sequence[Literal["Men", "Women"]], optional
        Filter for Men's or Women's tournament. Defaults to both.
    rounds : Sequence[int], optional
        If provided, only includes games from the specified rounds. Defaults to all rounds.
    mode : Literal["brier", "logloss"], optional
        The evaluation metric to use. "brier" for Brier score, "logloss" for log loss. Defaults to "brier".

    Returns
    -------
    score : float
        The resulting score from the historical tournament results, predictions, and chosen parameters.
    """
    scores = evaluate_stage1_submission_games(
        submission=submission,
        seasons=seasons,
        genders=genders,
        rounds=rounds,
        mode=mode,
    )
    return np.array(list(scores.values())).mean()


def evaluate_stage1_submission_games(
    submission: pd.DataFrame,
    seasons: Sequence[int] = (2021, 2022, 2023, 2024),
    genders: Sequence[Literal["Men", "Women"]] = ("Men", "Women"),
    rounds: Sequence[int] = (1, 2, 3, 4, 5, 6),
    mode: Literal["brier", "logloss"] = "brier",
) -> dict[str, float]:
    """
    Evaluate scores for each game in a submission.

    Parameters
    ----------
    submission : pd.DataFrame
        The submission DataFrame, containing forecasted probabilities.
    seasons : Sequence[int], optional
        The seasons to evaluate the submission for. Defaults to (2021, 2022, 2023, 2024).
    mode : Literal["brier", "logloss"], optional
        The evaluation metric to use. "brier" for Brier score, "logloss" for log loss. Defaults to "brier".

    Returns
    -------
    scores : dict[str, float]
        A dictionary with the resulting score from the historical tournament results, predictions, and chosen parameters
        for each game.
    """
    prediction_dict = submission.set_index("ID")["Pred"].to_dict()
    results_dict = _get_historical_results(
        seasons=seasons, genders=genders, rounds=rounds
    )
    match mode:
        case "brier":
            f = _brier
        case "logloss":
            f = _logloss
    return {gid: f(prediction_dict[gid], r) for gid, r in results_dict.items()}


def validate_submission_format(
    submission: pd.DataFrame, check_seasons: Sequence[int] = [2025]
) -> dict[str, bool]:
    """
    Validate that a submission is ready for submission.

    This function checks the format of the submission to ensure it meets the requirements of the competition.
    Use this to verify your submission before uploading it to Kaggle.

    Note that this is not endorsed by Kaggle and the function may have errors. Please reach out if you suspect
    there are any issues.

    Parameters
    ----------
    submission : pd.DataFrame
        The submission DataFrame to validate.
    check_seasons : Sequence[int], optional
        The seasons to validate the submission for. Defaults to [2025].

    Returns
    -------
    validation_results : dict[str, bool]
        A dictionary with the validation results.
    """
    checks = {
        "validation info": {
            "checking seasons": check_seasons,
            "columns": submission.columns.tolist(),
            "row count": len(submission),
        },
    }
    checks["correct columns"] = _check_columns(submission)
    checks["correct team order in ids"] = _check_id_team_order(submission)
    checks["correct row count"] = _check_sub_count(submission, check_seasons)
    return checks


def make_template_submission(seasons: Sequence[int]) -> pd.DataFrame:
    """
    Creates a template submission DataFrame for the specified seasons.

    Parameters
    ----------
    seasons : Sequence[int]
        The seasons for which to create a template submission.

    Returns
    -------
    template_submission : pd.DataFrame
        A template submission DataFrame with the correct columns and indices.
    """
    game_ids = []
    for s in seasons:
        m_teams, w_teams = _get_season_teams(s)
        for teams in (m_teams, w_teams):
            game_ids.extend(f"{s}_{a}_{b}" for a, b in combinations(teams, 2))
    sub = pd.DataFrame({"ID": game_ids, "Pred": np.full(len(game_ids), np.nan)})
    return sub


#########################################################
#      Private functions and tests below this point


@functools.cache
def _cached_csv_read(path: Path) -> pd.DataFrame:
    """cached read to improve performance"""
    return pd.read_csv(path)


# def _get_historical_results(
#     seasons: Sequence[int] = (2021, 2022, 2023, 2024),
# ) -> dict[str, int]:
#     """
#     parse a list of historical results to the expected kaggle result format in a dict
#     """
#     tourney_df = pd.concat(
#         [
#             _cached_csv_read(
#                 DEFAULT_COMPETITION_DATA_PATH / f"{mw}NCAATourneyDetailedResults.csv"
#             )
#             for mw in "MW"
#         ]
#     )
#     seeds_df = pd.concat(
#         [
#             _cached_csv_read(
#                 DEFAULT_COMPETITION_DATA_PATH / f"{mw}NCAATourneySeeds.csv"
#             )
#             for mw in "MW"
#         ]
#     )
#     # filter unneeded rows/columns
#     tourney_df = tourney_df.loc[
#         tourney_df["Season"].isin(seasons), ["Season", "WTeamID", "LTeamID"]
#     ].copy()
#     # add seeds to find play in games
#     for wl in "WL":
#         tourney_df = pd.merge(
#             left=tourney_df,
#             right=seeds_df.add_suffix(f"_{wl}")[
#                 [f"Season_{wl}", f"TeamID_{wl}", f"Seed_{wl}"]
#             ],
#             how="left",
#             left_on=["Season", f"{wl}TeamID"],
#             right_on=[f"Season_{wl}", f"TeamID_{wl}"],
#         ).copy()
#     # filter play-in games out of results
#     # play_in_game_mask = tourney_df["Seed_W"].map(
#     #     lambda x: x in ("a", "b")
#     # ) & tourney_df["Seed_L"].map(lambda x: x in ("a", "b"))
#     play_in_game_mask = tourney_df["Seed_W"].str.endswith(("a", "b")) & tourney_df["Seed_L"].str.endswith(("a", "b"))
#     kaggle_results = tourney_df[~play_in_game_mask].copy()
#     # get the game ids for all games - all results are 1
#     w_ids = (
#         kaggle_results["Season"].astype(str)
#         + "_"
#         + kaggle_results["WTeamID"].astype(str)
#         + "_"
#         + kaggle_results["LTeamID"].astype(str)
#     )
#     w_ids.to_list()
#     # flip game ids and results so that lowest team id is listed first
#     # dont_flip = (tourney_df["WTeamID"] < tourney_df["LTeamID"]).to_list()
#     dont_flip = (kaggle_results["WTeamID"] < kaggle_results["LTeamID"]).to_list()
#     return {wid if df else _swap_game_id(wid): df for wid, df in zip(w_ids, dont_flip)}


def _swap_game_id(game_id: str) -> str:
    """swaps team a and b in a game id"""
    parts = game_id.split("_")
    parts.append(parts.pop(1))
    return "_".join(parts)


def _brier(pred: float, result: int) -> float:
    """calculates brier score"""
    return (pred - result) ** 2


def _logloss(pred: float, result: int) -> float:
    """calculates log loss"""
    return (result * -np.log(pred)) + ((1 - result) * -np.log(1 - pred))


def _check_columns(submission: pd.DataFrame) -> bool:
    """checks that columns are ID and Pred"""
    return set(submission.columns) == {"ID", "Pred"}


def _check_id_team_order(submission: pd.DataFrame) -> bool:
    """checks that team A ID is greater than team B ID"""
    return (
        submission["ID"]
        .map(lambda gid: int((parts := gid.split("_"))[1]) < int(parts[2]))
        .any()
    )


def _check_sub_count(
    submission: pd.DataFrame,
    seasons: Sequence[int] = (2021, 2022, 2023, 2024),
) -> bool:
    """checks the number of rows in a submission"""
    team_counts = [_get_season_teams(s) for s in seasons]
    possible_match_counts = [
        _comb_count(len(m)) + _comb_count(len(w)) for m, w in team_counts
    ]
    expected_count = sum(possible_match_counts)
    return len(submission) == expected_count


def _comb_count(n: int) -> int:
    """calculates the number of unique pairs"""
    return int((n * (n - 1)) / 2)


def _get_season_teams(season: int) -> tuple[list[int], list[int]]:
    """gets the unique teams for both men and women in a given season"""
    m_conference_teams = _cached_csv_read(
        DEFAULT_COMPETITION_DATA_PATH / "MTeamConferences.csv"
    )
    w_conference_teams = _cached_csv_read(
        DEFAULT_COMPETITION_DATA_PATH / "WTeamConferences.csv"
    )
    return (
        m_conference_teams[m_conference_teams["Season"] == season]["TeamID"]
        .astype(int)
        .to_list(),
        w_conference_teams[w_conference_teams["Season"] == season]["TeamID"]
        .astype(int)
        .to_list(),
    )


#### tests ####


def _test_sample_evaluation() -> None:
    """make sure sample submissions return expected values"""
    expectations = {
        DEFAULT_COMPETITION_DATA_PATH / "SampleSubmissionStage1.csv": 0.25,
        # DEFAULT_COMPETITION_DATA_PATH / "SeedBenchmarkStage1.csv": 0.18767401129943503,
        DEFAULT_COMPETITION_DATA_PATH / "SeedBenchmarkStage1.csv": 0.1842045725646123,
    }
    for sample_path, expected_result in expectations.items():
        if not sample_path.exists():
            warnings.warn("{sample_path} does not exist... skipping tests")
        sample = _cached_csv_read(sample_path)
        seasons = sample["ID"].map(lambda x: int(x.split("_")[0])).unique().tolist()
        score = evaluate_stage1_submission(sample, seasons)
        assert np.isclose(score, expected_result), (
            f"tests not returning expected results: {score} != {expected_result}"
        )


_test_sample_evaluation()


def _test_submission_count() -> None:
    """make sure the row count checker agrees with provided samples"""
    paths = (
        DEFAULT_COMPETITION_DATA_PATH / "SampleSubmissionStage1.csv",
        DEFAULT_COMPETITION_DATA_PATH / "SampleSubmissionStage2.csv",
        DEFAULT_COMPETITION_DATA_PATH / "SeedBenchmarkStage1.csv",
    )
    for sample_path in paths:
        if not sample_path.exists():
            warnings.warn("{sample_path} does not exist... skipping tests")
        sample = _cached_csv_read(sample_path)
        seasons = sample["ID"].map(lambda x: int(x.split("_")[0])).unique().tolist()
        assert not _check_sub_count(sample.head(), seasons), (
            "something is wrong with count checker"
        )
        assert _check_sub_count(sample, seasons), (
            "something is wrong with count checker"
        )


_test_submission_count()


def _test_submission_team_order() -> None:
    """make sure order checker agrees with provided samples"""
    paths = (
        DEFAULT_COMPETITION_DATA_PATH / "SampleSubmissionStage1.csv",
        DEFAULT_COMPETITION_DATA_PATH / "SampleSubmissionStage2.csv",
        DEFAULT_COMPETITION_DATA_PATH / "SeedBenchmarkStage1.csv",
    )
    for sample_path in paths:
        if not sample_path.exists():
            warnings.warn("{sample_path} does not exist... skipping tests")
        sample = _cached_csv_read(sample_path)
        bad_sample = sample.copy()
        bad_sample["ID"] = bad_sample["ID"].map(_swap_game_id)

        assert _check_id_team_order(sample.head()), (
            "something is wrong with team order checker"
        )
        assert not _check_id_team_order(bad_sample.head()), (
            "something is wrong with team order checker"
        )


_test_submission_team_order()


def _test_submission_columns() -> None:
    """make sure column checker agrees with provided samples"""
    paths = (
        DEFAULT_COMPETITION_DATA_PATH / "SampleSubmissionStage1.csv",
        DEFAULT_COMPETITION_DATA_PATH / "SampleSubmissionStage2.csv",
        DEFAULT_COMPETITION_DATA_PATH / "SeedBenchmarkStage1.csv",
    )
    for sample_path in paths:
        if not sample_path.exists():
            warnings.warn("{sample_path} does not exist... skipping tests")
        sample = _cached_csv_read(sample_path)
        assert _check_columns(sample.head()), (
            "something is wrong with team order checker"
        )
        bad_sample = sample.copy()
        assert not _check_columns(bad_sample.rename(columns={"Pred": "P"}).head()), (
            "something is wrong with team order checker"
        )
        bad_sample["extra_column"] = True
        assert not _check_columns(bad_sample.head()), (
            "something is wrong with team order checker"
        )


_test_submission_columns()


def test_submission_creation() -> None:
    """creates a sample template and ensures it is consistent with checks"""
    stage1_seasons = (2021, 2022, 2023, 2024, 2025)
    stage1_sub = make_template_submission(stage1_seasons)
    assert _check_columns(stage1_sub)
    assert _check_id_team_order(stage1_sub)
    assert _check_sub_count(stage1_sub, stage1_seasons)


test_submission_creation()

### Troubleshooting `_get_historical_results()`

In [65]:
def _swap_game_id(game_id: str) -> str:
    """swaps team a and b in a game id"""
    parts = game_id.split("_")
    parts.append(parts.pop(1))
    return "_".join(parts)


seasons = (2021, 2022, 2023, 2024)

tourney_df = pd.concat(
    [
        pd.read_csv(
            DEFAULT_COMPETITION_DATA_PATH / f"{mw}NCAATourneyDetailedResults.csv"
        )
        for mw in "MW"
    ]
)
seeds_df = pd.concat(
    [
        pd.read_csv(DEFAULT_COMPETITION_DATA_PATH / f"{mw}NCAATourneySeeds.csv")
        for mw in "MW"
    ]
)

# filter unneeded rows/columns
tourney_df = tourney_df.loc[
    tourney_df["Season"].isin(seasons), ["Season", "WTeamID", "LTeamID"]
].copy()

# add seeds to find play in games
for wl in "WL":
    tourney_df = pd.merge(
        left=tourney_df,
        right=seeds_df.add_suffix(f"_{wl}")[
            [f"Season_{wl}", f"TeamID_{wl}", f"Seed_{wl}"]
        ],
        how="left",
        left_on=["Season", f"{wl}TeamID"],
        right_on=[f"Season_{wl}", f"TeamID_{wl}"],
    ).copy()

# filter play-in games out of results
# play_in_game_mask = tourney_df["Seed_W"].map(
#     lambda x: x in ("a", "b")
# ) & tourney_df["Seed_L"].map(lambda x: x in ("a", "b"))
# kaggle_results = tourney_df[~play_in_game_mask].copy()

# filter play-in games out of results
play_in_game_mask = tourney_df["Seed_W"].str.endswith(("a", "b")) & tourney_df[
    "Seed_L"
].str.endswith(("a", "b"))
kaggle_results = tourney_df[~play_in_game_mask].copy()

# get the game ids for all games - all results are 1
w_ids = (
    kaggle_results["Season"].astype(str)
    + "_"
    + kaggle_results["WTeamID"].astype(str)
    + "_"
    + kaggle_results["LTeamID"].astype(str)
)
w_ids.to_list()
# flip game ids and results so that lowest team id is listed first
dont_flip = (tourney_df["WTeamID"] < tourney_df["LTeamID"]).to_list()

results_dict = {
    wid if df else _swap_game_id(wid): df for wid, df in zip(w_ids, dont_flip)
}

In [47]:
tourney_df.head()

,Season,WTeamID,LTeamID,Season_W,TeamID_W,Seed_W,Season_L,TeamID_L,Seed_L
0,2021,1179,1455,2021,1179,X11a,2021,1455,X11b
1,2021,1313,1111,2021,1313,X16b,2021,1111,X16a
2,2021,1411,1291,2021,1411,W16b,2021,1291,W16a
3,2021,1417,1277,2021,1417,W11b,2021,1277,W11a
4,2021,1116,1159,2021,1116,Z03,2021,1159,Z14


In [72]:
tourney_df[
    tourney_df["Seed_W"].map(lambda x: x in ("a", "b"))
    & tourney_df["Seed_L"].map(lambda x: x in ("a", "b"))
]

,Season,WTeamID,LTeamID,Season_W,TeamID_W,Seed_W,Season_L,TeamID_L,Seed_L


In [64]:
tourney_df[
    tourney_df["Seed_W"].str.endswith(("a", "b"))
    & tourney_df["Seed_L"].str.endswith(("a", "b"))
]

,Season,WTeamID,LTeamID,Season_W,TeamID_W,Seed_W,Season_L,TeamID_L,Seed_L
0,2021,1179,1455,2021,1179,X11a,2021,1455,X11b
1,2021,1313,1111,2021,1313,X16b,2021,1111,X16a
2,2021,1411,1291,2021,1411,W16b,2021,1291,W16a
3,2021,1417,1277,2021,1417,W11b,2021,1277,W11a
66,2022,1231,1461,2022,1231,W12a,2022,1461,W12b
67,2022,1411,1394,2022,1411,Y16b,2022,1394,Y16a
68,2022,1323,1353,2022,1323,X11a,2022,1353,X11b
69,2022,1460,1136,2022,1460,Z16b,2022,1136,Z16a
133,2023,1338,1280,2023,1338,Y11b,2023,1280,Y11a
134,2023,1394,1369,2023,1394,X16b,2023,1369,X16a


## `SampleSubmissionStage1.csv` - Score: 0.25

In [53]:
sub = pd.read_csv(DEFAULT_COMPETITION_DATA_PATH / "SampleSubmissionStage1.csv")

In [41]:
evaluate_stage1_submission?

Signature:
evaluate_stage1_submission(
    submission: pandas.core.frame.DataFrame,
    seasons: Sequence[int] = (2021, 2022, 2023, 2024),
    genders: Sequence[Literal['Men', 'Women']] = ('Men', 'Women'),
    rounds: Sequence[int] = (1, 2, 3, 4, 5, 6),
    mode: Literal['brier', 'logloss'] = 'brier',
) -> float
Docstring:
Evaluate a Stage 1 submission for a forecasting competition.

Parameters
----------
submission : pd.DataFrame
    The submission DataFrame, containing forecasted probabilities.
seasons : Sequence[int], optional
    The seasons to evaluate the submission for. Defaults to (2021, 2022, 2023, 2024).
genders : Sequence[Literal["Men", "Women"]], optional
    Filter for Men's or Women's tournament. Defaults to both.
rounds : Sequence[int], optional
    If provided, only includes games from the specified rounds. Defaults to all rounds.
mode : Literal["brier", "logloss"], optional
    The evaluation metric to use. "brier" for Brier score, "logloss" for log loss. Defaults to "

In [ ]:
display_submission_results(sub)

In [42]:
evaluate_stage1_submission(sub)

0.25

In [43]:
years = [2021, 2022, 2023, 2024]
results = {year: evaluate_stage1_submission(sub, seasons=[year]) for year in years}

# Print results
for year, score in results.items():
    print(f"Year {year}: {score:.6f}")

Year 2021: 0.250000
Year 2022: 0.250000
Year 2023: 0.250000
Year 2024: 0.250000


In [44]:
genders = ["Men", "Women"]
results = {
    gender: evaluate_stage1_submission(sub, genders=[gender]) for gender in genders
}

# Print results
for gender, score in results.items():
    print(f"Gender {gender}: {score:.6f}")

Gender Men: 0.250000
Gender Women: 0.250000


In [45]:
rounds = [1, 2, 3, 4, 5, 6]
results = {round: evaluate_stage1_submission(sub, rounds=[round]) for round in rounds}

# Print results
for round, score in results.items():
    print(f"Round {round}: {score:.6f}")

Round 1: 0.250000
Round 2: 0.250000
Round 3: 0.250000
Round 4: 0.250000
Round 5: 0.250000
Round 6: 0.250000


In [31]:
validate_submission_format(sub, check_seasons=[2021, 2022, 2023, 2024])

{'validation info': {'checking seasons': [2021, 2022, 2023, 2024],
  'columns': ['ID', 'Pred'],
  'row count': 507108},
 'correct columns': True,
 'correct team order in ids': True,
 'correct row count': True}

In [ ]:
evaluate_stage1_submission_games(sub, [2024])

## `SeedBenchmarkStage1.csv` - Score: 0.1842045725646123

In [57]:
sub = pd.read_csv(DEFAULT_COMPETITION_DATA_PATH / "SeedBenchmarkStage1.csv")

In [ ]:
display_submission_results(sub)

In [84]:
years = [2021, 2022, 2023, 2024]
genders = ["Men", "Women"]
rounds = [1]

results = {
    "Total Score": evaluate_stage1_submission(sub),
    "By Year": {
        year: evaluate_stage1_submission(sub, seasons=[year]) for year in years
    },
    "By Gender": {
        gender: evaluate_stage1_submission(sub, genders=[gender]) for gender in genders
    },
    "By Year, Gender": {
        (year, gender): evaluate_stage1_submission(
            sub, seasons=[year], genders=[gender]
        )
        for year in years
        for gender in genders
    },
    "By Year, Gender, Round": {
        (year, gender, rnd): evaluate_stage1_submission(
            sub, seasons=[year], genders=[gender], rounds=[rnd]
        )
        for year in years
        for gender in genders
        for rnd in rounds
    },
}

for key, value in results.items():
    print(f"{key}: {value}")

Total Score: 0.1842045725646123
By Year: {2021: 0.1897312, 2022: 0.1921190476190476, 2023: 0.18839999999999998, 2024: 0.16661190476190474}
By Gender: {'Men': 0.20775617529880477, 'Women': 0.16074642857142857}
By Year, Gender: {(2021, 'Men'): 0.22218225806451614, (2021, 'Women'): 0.15779523809523813, (2022, 'Men'): 0.2134666666666667, (2022, 'Women'): 0.1707714285714286, (2023, 'Men'): 0.20577142857142863, (2023, 'Women'): 0.17102857142857147, (2024, 'Men'): 0.18983333333333333, (2024, 'Women'): 0.14339047619047618}
By Year, Gender, Round: {(2021, 'Men', 1): 0.21160967741935485, (2021, 'Women', 1): 0.12587500000000001, (2022, 'Men', 1): 0.1765, (2022, 'Women', 1): 0.14650000000000002, (2023, 'Men', 1): 0.174625, (2023, 'Women', 1): 0.137125, (2024, 'Men', 1): 0.18962500000000002, (2024, 'Women', 1): 0.09587499999999999}


In [79]:
genders = ["Men"]
if set(genders) == {"Men", "Women"}:
    gender_prefix = "MW"
elif "Men" in genders:
    gender_prefix = "M"
else:
    gender_prefix = "W"

print(gender_prefix)

M


In [68]:
evaluate_stage1_submission(sub, genders=["Women"])

0.16074642857142857

In [72]:
evaluate_stage1_submission(sub)

0.1842045725646123

In [49]:
years = [2021, 2022, 2023, 2024]
results = {year: evaluate_stage1_submission(sub, [year]) for year in years}

# Print results
for year, score in results.items():
    print(f"Year {year}: {score:.6f}")

Year 2021: 0.189731
Year 2022: 0.192119
Year 2023: 0.188400
Year 2024: 0.166612


In [50]:
genders = ["Men", "Women"]
results = {
    gender: evaluate_stage1_submission(sub, genders=[gender]) for gender in genders
}

# Print results
for gender, score in results.items():
    print(f"Gender {gender}: {score:.6f}")

Gender Men: 0.160746
Gender Women: 0.160746


In [51]:
rounds = [1, 2, 3, 4, 5, 6]
results = {round: evaluate_stage1_submission(sub, rounds=[round]) for round in rounds}

# Print results
for round, score in results.items():
    print(f"Round {round}: {score:.6f}")

Round 1: 0.157004
Round 2: 0.192163
Round 3: 0.242589
Round 4: 0.223506
Round 5: 0.224613
Round 6: 0.218800


In [76]:
validate_submission_format(sub, check_seasons=[2021, 2022, 2023, 2024])

{'validation info': {'checking seasons': [2021, 2022, 2023, 2024],
  'columns': ['ID', 'Pred'],
  'row count': 507108},
 'correct columns': True,
 'correct team order in ids': True,
 'correct row count': True}

In [ ]:
evaluate_stage1_submission_games(sub, [2024])

## `Round1_SeedProbabilities.csv` - Score: 0.1999782330739834

| Group | Score |
| :-------- | -------: |
| Year 2021: | 0.208775 |
| Year 2022: | 0.201173 |
| Year 2023: | 0.203056 |
| Year 2024: | 0.186979 |
| Gender Men: | 0.217651 |
| Gender Women: | 0.182375 |
| **Round 1**: | **0.151330** |
| Round 2: | 0.250000 |
| Round 3: | 0.250000 |
| Round 4: | 0.250000 |
| Round 5: | 0.250000 |
| Round 6: | 0.250000 |

In [85]:
sub = pd.read_csv(SUBMISSION_DATA_PATH / "Round1_SeedProbabilities.csv")

In [86]:
validate_submission_format(sub, check_seasons=[2021, 2022, 2023, 2024])

{'validation info': {'checking seasons': [2021, 2022, 2023, 2024],
  'columns': ['ID', 'Pred'],
  'row count': 507108},
 'correct columns': True,
 'correct team order in ids': True,
 'correct row count': True}

In [87]:
evaluate_stage1_submission(sub)

0.1999782330739834

In [88]:
years = [2021, 2022, 2023, 2024]
results = {year: evaluate_stage1_submission(sub, [year]) for year in years}

# Print results
for year, score in results.items():
    print(f"Year {year}: {score:.6f}")

Year 2021: 0.208775
Year 2022: 0.201173
Year 2023: 0.203056
Year 2024: 0.186979


In [89]:
genders = ["Men", "Women"]
results = {
    gender: evaluate_stage1_submission(sub, genders=[gender]) for gender in genders
}

# Print results
for gender, score in results.items():
    print(f"Gender {gender}: {score:.6f}")

Gender Men: 0.217651
Gender Women: 0.182375


In [90]:
rounds = [1, 2, 3, 4, 5, 6]
results = {round: evaluate_stage1_submission(sub, rounds=[round]) for round in rounds}

# Print results
for round, score in results.items():
    print(f"Round {round}: {score:.6f}")

Round 1: 0.151330
Round 2: 0.250000
Round 3: 0.250000
Round 4: 0.250000
Round 5: 0.250000
Round 6: 0.250000


## `Round1_BetExplorer.csv`

In [91]:
sub = pd.read_csv(SUBMISSION_DATA_PATH / "Round1_BetExplorer.csv")

In [92]:
validate_submission_format(sub, check_seasons=[2021, 2022, 2023, 2024])

{'validation info': {'checking seasons': [2021, 2022, 2023, 2024],
  'columns': ['ID', 'Pred'],
  'row count': 507108},
 'correct columns': True,
 'correct team order in ids': True,
 'correct row count': True}

In [93]:
evaluate_stage1_submission(sub)

0.23604547024908124

In [94]:
years = [2021, 2022, 2023, 2024]
results = {year: evaluate_stage1_submission(sub, [year]) for year in years}

# Print results
for year, score in results.items():
    print(f"Year {year}: {score:.6f}")

Year 2021: 0.238481
Year 2022: 0.233818
Year 2023: 0.232946
Year 2024: 0.238956


In [95]:
genders = ["Men", "Women"]
results = {
    gender: evaluate_stage1_submission(sub, genders=[gender]) for gender in genders
}

# Print results
for gender, score in results.items():
    print(f"Gender {gender}: {score:.6f}")

Gender Men: 0.222035
Gender Women: 0.250000


In [96]:
rounds = [1, 2, 3, 4, 5, 6]
results = {round: evaluate_stage1_submission(sub, rounds=[round]) for round in rounds}

# Print results
for round, score in results.items():
    print(f"Round {round}: {score:.6f}")

Round 1: 0.222474
Round 2: 0.250000
Round 3: 0.250000
Round 4: 0.250000
Round 5: 0.250000
Round 6: 0.250000


## `Round1_538Ratings.csv`

In [110]:
sub = pd.read_csv(SUBMISSION_DATA_PATH / "Round1_538Ratings.csv")

In [111]:
validate_submission_format(sub, check_seasons=[2021, 2022, 2023, 2024])

{'validation info': {'checking seasons': [2021, 2022, 2023, 2024],
  'columns': ['ID', 'Pred'],
  'row count': 507108},
 'correct columns': True,
 'correct team order in ids': True,
 'correct row count': True}

In [112]:
evaluate_stage1_submission(sub)

0.198055385979984

In [113]:
years = [2021, 2022, 2023, 2024]
results = {year: evaluate_stage1_submission(sub, [year]) for year in years}

# Print results
for year, score in results.items():
    print(f"Year {year}: {score:.6f}")

Year 2021: 0.176665
Year 2022: 0.180161
Year 2023: 0.185226
Year 2024: 0.250000


In [114]:
genders = ["Men", "Women"]
results = {
    gender: evaluate_stage1_submission(sub, genders=[gender]) for gender in genders
}

# Print results
for gender, score in results.items():
    print(f"Gender {gender}: {score:.6f}")

Gender Men: 0.221293
Gender Women: 0.174909


In [115]:
rounds = [1, 2, 3, 4, 5, 6]
results = {round: evaluate_stage1_submission(sub, rounds=[round]) for round in rounds}

# Print results
for round, score in results.items():
    print(f"Round {round}: {score:.6f}")

Round 1: 0.183628
Round 2: 0.205336
Round 3: 0.235543
Round 4: 0.179511
Round 5: 0.264376
Round 6: 0.183065


## `vilnius_ncaa_raddar_stage1.csv` - Score: 0.15668069368660015

In [7]:
sub = pd.read_csv(SUBMISSION_DATA_PATH / "vilnius_ncaa_raddar_stage1.csv")

In [8]:
validate_submission_format(sub, check_seasons=[2021, 2022, 2023, 2024])

{'validation info': {'checking seasons': [2021, 2022, 2023, 2024],
  'columns': ['ID', 'Pred'],
  'row count': 507108},
 'correct columns': True,
 'correct team order in ids': True,
 'correct row count': True}

In [9]:
evaluate_stage1_submission(sub)

0.15668069368660015

In [10]:
years = [2021, 2022, 2023, 2024]
results = {year: evaluate_stage1_submission(sub, [year]) for year in years}

# Print results
for year, score in results.items():
    print(f"Year {year}: {score:.6f}")

Year 2021: 0.158432
Year 2022: 0.164807
Year 2023: 0.165834
Year 2024: 0.137663


In [11]:
genders = ["Men", "Women"]
results = {
    gender: evaluate_stage1_submission(sub, genders=[gender]) for gender in genders
}

# Print results
for gender, score in results.items():
    print(f"Gender {gender}: {score:.6f}")

Gender Men: 0.187646
Gender Women: 0.125838


In [12]:
rounds = [1, 2, 3, 4, 5, 6]
results = {round: evaluate_stage1_submission(sub, rounds=[round]) for round in rounds}

# Print results
for round, score in results.items():
    print(f"Round {round}: {score:.6f}")

Round 1: 0.140631
Round 2: 0.156250
Round 3: 0.219883
Round 4: 0.149652
Round 5: 0.203080
Round 6: 0.104862


## `vilnius_ncaa_raddar_stage1_v2.csv` - Score: 0.15642918697249275

In [13]:
sub = pd.read_csv(SUBMISSION_DATA_PATH / "vilnius_ncaa_raddar_stage1_v2.csv")

In [14]:
validate_submission_format(sub, check_seasons=[2021, 2022, 2023, 2024])

{'validation info': {'checking seasons': [2021, 2022, 2023, 2024],
  'columns': ['ID', 'Pred'],
  'row count': 507108},
 'correct columns': True,
 'correct team order in ids': True,
 'correct row count': True}

In [15]:
evaluate_stage1_submission(sub)

0.15642918697249275

In [16]:
years = [2021, 2022, 2023, 2024]
results = {year: evaluate_stage1_submission(sub, [year]) for year in years}

# Print results
for year, score in results.items():
    print(f"Year {year}: {score:.6f}")

Year 2021: 0.159302
Year 2022: 0.164792
Year 2023: 0.164494
Year 2024: 0.137151


In [17]:
genders = ["Men", "Women"]
results = {
    gender: evaluate_stage1_submission(sub, genders=[gender]) for gender in genders
}

# Print results
for gender, score in results.items():
    print(f"Gender {gender}: {score:.6f}")

Gender Men: 0.186646
Gender Women: 0.126333


In [18]:
rounds = [1, 2, 3, 4, 5, 6]
results = {round: evaluate_stage1_submission(sub, rounds=[round]) for round in rounds}

# Print results
for round, score in results.items():
    print(f"Round {round}: {score:.6f}")

Round 1: 0.139945
Round 2: 0.156107
Round 3: 0.221243
Round 4: 0.147916
Round 5: 0.205525
Round 6: 0.104383


## `vilnius_ncaa_raddar_stage1_v3.csv` - Score: 0.15647033213589334

In [20]:
sub = pd.read_csv(SUBMISSION_DATA_PATH / "vilnius_ncaa_raddar_stage1_v3.csv")

In [21]:
validate_submission_format(sub, check_seasons=[2021, 2022, 2023, 2024])

{'validation info': {'checking seasons': [2021, 2022, 2023, 2024],
  'columns': ['ID', 'Pred'],
  'row count': 507108},
 'correct columns': True,
 'correct team order in ids': True,
 'correct row count': True}

In [22]:
evaluate_stage1_submission(sub)

0.15647033213589334

In [23]:
years = [2021, 2022, 2023, 2024]
results = {year: evaluate_stage1_submission(sub, [year]) for year in years}

# Print results
for year, score in results.items():
    print(f"Year {year}: {score:.6f}")

Year 2021: 0.161855
Year 2022: 0.158562
Year 2023: 0.165258
Year 2024: 0.140250


In [24]:
genders = ["Men", "Women"]
results = {
    gender: evaluate_stage1_submission(sub, genders=[gender]) for gender in genders
}

# Print results
for gender, score in results.items():
    print(f"Gender {gender}: {score:.6f}")

Gender Men: 0.186116
Gender Women: 0.126942


In [25]:
rounds = [1, 2, 3, 4, 5, 6]
results = {round: evaluate_stage1_submission(sub, rounds=[round]) for round in rounds}

# Print results
for round, score in results.items():
    print(f"Round {round}: {score:.6f}")

Round 1: 0.139025
Round 2: 0.155724
Round 3: 0.221319
Round 4: 0.152257
Round 5: 0.206302
Round 6: 0.122882


## `vilnius_ncaa_raddar_stage1_v4.csv` - Score: 0.1547444962441538

In [26]:
sub = pd.read_csv(SUBMISSION_DATA_PATH / "vilnius_ncaa_raddar_stage1_v4.csv")

In [27]:
validate_submission_format(sub, check_seasons=[2021, 2022, 2023, 2024])

{'validation info': {'checking seasons': [2021, 2022, 2023, 2024],
  'columns': ['ID', 'Pred'],
  'row count': 507108},
 'correct columns': True,
 'correct team order in ids': True,
 'correct row count': True}

In [28]:
evaluate_stage1_submission(sub)

0.1547444962441538

In [29]:
years = [2021, 2022, 2023, 2024]
results = {year: evaluate_stage1_submission(sub, [year]) for year in years}

# Print results
for year, score in results.items():
    print(f"Year {year}: {score:.6f}")

Year 2021: 0.161901
Year 2022: 0.155695
Year 2023: 0.163360
Year 2024: 0.138079


In [30]:
genders = ["Men", "Women"]
results = {
    gender: evaluate_stage1_submission(sub, genders=[gender]) for gender in genders
}

# Print results
for gender, score in results.items():
    print(f"Gender {gender}: {score:.6f}")

Gender Men: 0.182743
Gender Women: 0.126857


In [31]:
rounds = [1, 2, 3, 4, 5, 6]
results = {round: evaluate_stage1_submission(sub, rounds=[round]) for round in rounds}

# Print results
for round, score in results.items():
    print(f"Round {round}: {score:.6f}")

Round 1: 0.136403
Round 2: 0.159136
Round 3: 0.212569
Round 4: 0.144808
Round 5: 0.203233
Round 6: 0.149276


## `vilnius_ncaa_raddar_stage1_v5.csv` - Score: 0.15569742018088573

In [32]:
sub = pd.read_csv(SUBMISSION_DATA_PATH / "vilnius_ncaa_raddar_stage1_v5.csv")

In [33]:
validate_submission_format(sub, check_seasons=[2021, 2022, 2023, 2024])

{'validation info': {'checking seasons': [2021, 2022, 2023, 2024],
  'columns': ['ID', 'Pred'],
  'row count': 507108},
 'correct columns': True,
 'correct team order in ids': True,
 'correct row count': True}

In [34]:
evaluate_stage1_submission(sub)

0.15569742018088573

In [35]:
years = [2021, 2022, 2023, 2024]
results = {year: evaluate_stage1_submission(sub, [year]) for year in years}

# Print results
for year, score in results.items():
    print(f"Year {year}: {score:.6f}")

Year 2021: 0.161265
Year 2022: 0.157631
Year 2023: 0.165635
Year 2024: 0.138303


In [36]:
genders = ["Men", "Women"]
results = {
    gender: evaluate_stage1_submission(sub, genders=[gender]) for gender in genders
}

# Print results
for gender, score in results.items():
    print(f"Gender {gender}: {score:.6f}")

Gender Men: 0.185501
Gender Women: 0.126012


In [37]:
rounds = [1, 2, 3, 4, 5, 6]
results = {round: evaluate_stage1_submission(sub, rounds=[round]) for round in rounds}

# Print results
for round, score in results.items():
    print(f"Round {round}: {score:.6f}")

Round 1: 0.138096
Round 2: 0.157239
Round 3: 0.220155
Round 4: 0.148993
Round 5: 0.200275
Round 6: 0.114082


## `Seed_Elo_Odds.csv`

In [38]:
sub = pd.read_csv(SUBMISSION_DATA_PATH / "Seed_Elo_Odds.csv")

In [39]:
validate_submission_format(sub, check_seasons=[2025])

{'validation info': {'checking seasons': [2025],
  'columns': ['ID', 'Pred'],
  'row count': 131407},
 'correct columns': True,
 'correct team order in ids': True,
 'correct row count': True}